In [ ]:
# import numpy as np
# import pandas as pd
# import pyvinecopulib as pv
# from itertools import product

# def samplingorder_initial(a):
#     dimen = a.shape[0]  # dimension of data.
#     order = pd.DataFrame(
#         columns=["node", "l", "r", "tree"]
#     )  # dataframe for vine copula structure.
#     s = 0
#     for i in list(range(dimen - 1)):
#         for k in list(range(dimen - 1 - i)):
#             ak = a[:, k]  # edges
#             akn = np.array([ak[-1 - k], ak[i]]).astype(int)
#             if i == 0:
#                 single_row_values = {
#                     "node": list(akn),
#                     "l": akn[0],
#                     "r": akn[1],
#                     "tree": i,
#                 }
#             else:
#                 single_row_values = {
#                     "node": list(akn) + ["|"] + list((ak.astype(int)[:i])[::-1]),
#                     "l": list(akn),
#                     "r": list((ak.astype(int)[:i])[::-1]),
#                     "tree": i,
#                 }

#             order.loc[s] = single_row_values
#             s = s + 1
#     combinations = list(
#         product([True, False], repeat=dimen - 1)
#     )  # all diffferent sampling routes.
#     sortingorder = []
#     for q in combinations:
#         a = np.empty((dimen, dimen))
#         a[:] = np.nan
#         order["used"] = 0
#         for i in list(range(dimen - 1))[::-1]:
#             k1 = sorted(
#                 np.array(
#                     order[(order.tree == i) & (order["used"] == 0)].node.iloc[0][:2]
#                 ).astype(int),
#                 reverse=q[i],
#             )
#             order.loc[(order["tree"] == i) & (order["used"] == 0), "used"] = 1
#             ii = dimen - 2 - i
#             a[i : dimen - ii, ii] = k1
#             s = k1[-1]
#             for j in list(range(0, i))[::-1]:
#                 orde = order[(order.tree == j) & (order["used"] == 0)]
#                 for k in range(len(orde)):
#                     arr = np.array(orde.node.iloc[k][:2]).astype(int)
#                     if np.isin(s, arr) == True:
#                         inde = orde.iloc[k].name
#                         a[j, ii] = arr[arr != s][0]
#                         order.loc[inde, "used"] = 1
#         a[0, dimen - 1] = a[0, dimen - 2]
#         sortingorder.append(list(np.diag(a[::-1])[::-1]))  # add unique sampling orders
#     return np.array(sortingorder)

# def samplingorder_numpy(a):
#     """
#     Provides all the different sampling orders that are compatible with a given structure

#     Arguments:
#         *a* : The vine tree structure provided as a triangular matrix, composed of integers. The integer refers to different variables depending on which column the variable was in u1, where the first column is 0 and the second column is 1, etc.


#     Returns:
#      *sortingorder* :  A list of the different sampling orders available for the fitted vine-copula
#      """

#     dimen = a.shape[0]
#     order = []  # will hold dicts like {"node": [...], "l": ..., "r": ..., "tree": ..., "used": 0}
#     s = 0

#     # Build "order" structure
#     for i in range(dimen - 1):
#         for k in range(dimen - 1 - i):
#             ak = a[:, k]
#             akn = np.array([ak[-1 - k], ak[i]], dtype=int)

#             if i == 0:
#                 row = {
#                     "node": list(int(x) for x in akn),
#                     "l": int(akn[0]),
#                     "r": int(akn[1]),
#                     "tree": i,
#                     "used": 0,
#                 }
#             else:
#                 row = {
#                     "node": list(int(x) for x in akn) + ["|"] + list(int(x) for x in ak.astype(int)[:i][::-1]),
#                     "l": list(int(x) for x in akn),
#                     "r": list(int(x) for x in ak.astype(int)[:i][::-1]),
#                     "tree": i,
#                     "used": 0,
#                 }
#             order.append(row)
#             s += 1

#     # All possible sampling routes
#     combinations = list(product([True, False], repeat=dimen - 1))
#     sortingorder = []

#     for q in combinations:
#         mat = np.full((dimen, dimen), np.nan)
#         # reset "used" flags
#         for row in order:
#             row["used"] = 0

#         for i in reversed(range(dimen - 1)):
#             # pick first unused edge in this tree
#             candidate = next(row for row in order if row["tree"] == i and row["used"] == 0)
#             k1 = sorted(np.array(candidate["node"][:2], dtype=int), reverse=q[i])
#             candidate["used"] = 1

#             ii = dimen - 2 - i
#             mat[i : dimen - ii, ii] = k1
#             s = k1[-1]

#             # look at lower trees for matching edges
#             for j in reversed(range(i)):
#                 for row in order:
#                     if row["tree"] == j and row["used"] == 0:
#                         arr = np.array(row["node"][:2], dtype=int)
#                         if s in arr:
#                             mat[j, ii] = arr[arr != s][0]
#                             row["used"] = 1

#         mat[0, dimen - 1] = mat[0, dimen - 2]
#         sortingorder.append(list(np.diag(mat[::-1])[::-1]))

#     return np.array(sortingorder)


# class Edge:
#     def __init__(self, node, l, r, tree):
#         self.node = node              # list[int]
#         self.l = l                    # int
#         self.r = r                    # int
#         self.tree = tree              # int
#         self.used = False             # bool


# def samplingorder_numpy2(a):
#     dimen = a.shape[0]
#     order = []

#     # Build order
#     for i in range(dimen - 1):
#         for k in range(dimen - 1 - i):
#             ak = a[:, k]
#             akn = [int(ak[-1 - k]), int(ak[i])]

#             if i == 0:
#                 node = akn
#             else:
#                 # no more ["|"], just append conditioning set
#                 node = akn + list(ak[:i][::-1].astype(int))

#             edge = Edge(node=node, l=akn[0], r=akn[1], tree=i)
#             order.append(edge)

#     # All sampling routes
#     combinations = list(product([True, False], repeat=dimen - 1))
#     sortingorder = []

#     for q in combinations:
#         mat = [[None for _ in range(dimen)] for _ in range(dimen)]

#         # reset used flags
#         for edge in order:
#             edge.used = False

#         for i in reversed(range(dimen - 1)):
#             # pick first unused edge in this tree
#             candidate = None
#             for edge in order:
#                 if edge.tree == i and not edge.used:
#                     candidate = edge
#                     break

#             k1 = sorted(candidate.node[:2], reverse=q[i])
#             candidate.used = True

#             ii = dimen - 2 - i
#             # filling with k1
#             for row in range(i, dimen - ii):
#                 mat[row][ii] = k1[row - i]
#             s = k1[-1]

#             # look at lower trees for matching edges
#             for j in reversed(range(i)):
#                 for edge in order:
#                     if edge.tree == j and not edge.used:
#                         arr = edge.node[:2]
#                         if s in arr:
#                             mat[j][ii] = arr[0] if arr[1] == s else arr[1]
#                             edge.used = True

#         mat[0][dimen - 1] = mat[0][dimen - 2]

#         # explicit diagonal
#         diag = []
#         for idx in range(dimen):
#             diag.append(mat[dimen - 1 - idx][idx])
#         sortingorder.append(diag[::-1])

#     return np.array(sortingorder)

# def samplingorder_no_numpy(M):
#     """
#     NumPy-free logic for computing all sampling orders from an R-vine structure matrix M.
#     Records the FIRST element of the oriented pair per tree; uses the SECOND for descent.

#     M: list[list[int]] (square, lower-triangular R-vine structure matrix as returned by pyvinecopulib.get_matrix()).
#     Returns: np.ndarray of shape (2^(d-1), d) with dtype=int.
#     """
#     # --- dimensions & basic check ---
#     d = len(M)

#     # Accessor for the conditioned pair at tree t, column k
#     # Matches your earlier construction: akn = [ak[-1-k], ak[i]] with i=t, k=k
#     def pair(t, k):
#         u = M[d - 1 - k][k]  # anti-diagonal element of column k
#         v = M[t][k]          # row t element of column k
#         return u, v

#     # --- adjacency per tree: node -> list of edge indices k where node is in pair(t,k) ---
#     adj = []
#     for t in range(d - 1):
#         mp = {}
#         for k in range(d - 1 - t):
#             u, v = pair(t, k)
#             mp.setdefault(u, []).append(k)
#             mp.setdefault(v, []).append(k)
#         adj.append(mp)

#     # Enumerate routes in the same order as the NumPy version
#     routes = product([True, False], repeat=d - 1)
#     all_orders = []

#     for q in routes:
#         # used[t][k] indicates whether edge k of tree t has been consumed
#         used = [[False] * (d - 1 - t) for t in range(d - 1)]
#         seq = [None] * d  # final sampling order (length d)

#         # Process trees from top (t = d-2) down to 0
#         for t in range(d - 2, -1, -1):
#             # first unused edge in tree t
#             k0 = None
#             for k in range(len(used[t])):
#                 if not used[t][k]:
#                     k0 = k
#                     break
#             if k0 is None:
#                 raise RuntimeError("No unused edge found in current tree — structure inconsistent?")

#             u, v = pair(t, k0)

#             # Orientation: emulate sorted((u,v), reverse=q[t]).
#             # q[t] = False -> (a,b)=(min,max); q[t] = True -> (a,b)=(max,min)
#             if q[t]:
#                 a_val, b_val = (max(u, v), min(u, v))
#             else:
#                 a_val, b_val = (min(u, v), max(u, v))

#             used[t][k0] = True

#             # Column index ii and recording:
#             # Record the FIRST element of the oriented pair at position ii
#             ii = d - 2 - t
#             seq[ii] = a_val

#             # Use SECOND element for descent/marking of incident edges in lower trees
#             s = b_val
#             for j in range(t - 1, -1, -1):
#                 found = False
#                 for k in adj[j].get(s, ()):
#                     if not used[j][k]:
#                         used[j][k] = True
#                         found = True
#                         break
#                 if not found:
#                     raise RuntimeError(f"No unused edge in tree {j} incident to node {s}.")

#         # Final element equals the previous, matching the original behavior
#         seq[d - 1] = seq[d - 2]
#         all_orders.append(seq)

#     return np.array(all_orders)


# d = 6
# seed = 12
# rvs = pv.RVineStructure.simulate(d, seeds=[seed])
# a = rvs.matrix
# m = [[int(x) for x in a[i, :]] for i in range(a.shape[0])]
# # assert equivalence
# assert np.array_equal(samplingorder_initial(a), samplingorder_numpy2(a))
# assert np.array_equal(samplingorder_numpy(a), samplingorder_numpy2(a))
# assert np.array_equal(samplingorder_numpy(a), samplingorder_no_numpy(m))

AssertionError: 

In [ ]:
# print(
#   np.array(sorted(samplingorder_numpy(a).tolist())[:3]),
#   "\n",
#   np.array(sorted(samplingorder_no_numpy(m).tolist())[:3]),
# )

[[1. 3. 6. 2. 4. 5.]
 [1. 4. 6. 2. 3. 5.]
 [1. 4. 6. 2. 5. 3.]] 
 [[3 3 2 1 1 1]
 [3 3 2 1 6 6]
 [3 3 2 2 2 2]]
